In [2]:
import io
import getpass
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

pwd = getpass.getpass("PostgreSQL password: ")
url = URL.create(
    "postgresql+psycopg2",
    username="postgres",
    password=pwd,
    host="localhost",
    port=10038,
    database="malaysia_staple_prices",
)
engine = create_engine(url)

with engine.connect() as conn:
    print(conn.execute(text("SELECT current_database(), version()")).fetchone())

PostgreSQL password:  ········


('malaysia_staple_prices', 'PostgreSQL 18.1 on x86_64-windows, compiled by msvc-19.44.35221, 64-bit')


In [3]:
raw = Path("../data/raw")

item = pd.read_parquet(raw / "lookup_item.parquet")
premise = pd.read_parquet(raw / "lookup_premise.parquet")

item = item[item["item_code"] != -1]
premise = premise[premise["premise_code"] != -1].copy()
premise["premise_type"] = premise["premise_type"].str.strip()

basket = pd.read_csv("../docs/basket_items.csv")

with engine.begin() as conn:
    conn.execute(text("TRUNCATE basket_items, dim_item, dim_premise"))
    item.to_sql("dim_item", conn, if_exists="append", index=False)
    premise.to_sql("dim_premise", conn, if_exists="append", index=False)
    basket[["item_code"]].to_sql("basket_items", conn, if_exists="append", index=False)

with engine.connect() as conn:
    for t in ["dim_item", "dim_premise", "basket_items"]:
        print(t, conn.execute(text(f"SELECT COUNT(*) FROM {t}")).scalar())

dim_item 796
dim_premise 3928
basket_items 21
